# Day 09. Exercise 00
# Regularization

## 0. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/dayofweek.csv')
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

In [4]:
def cross_val_scores(model, X, y, n_splits=10):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=False)
    train_accs, val_accs = [], []
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_tr, y_tr)
        train_accs.append(model_clone.score(X_tr, y_tr))
        val_accs.append(model_clone.score(X_val, y_val))
        print(f"train - {train_accs[-1]:.5f}   |   valid - {val_accs[-1]:.5f}")
    mean_acc = np.mean(val_accs)
    std_acc = np.std(val_accs)
    print(f"Average accuracy on crossval is {mean_acc:.5f}")
    print(f"Std is {std_acc:.5f}\n")
    return mean_acc, std_acc, val_accs

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [5]:
%%time
logreg_baseline = LogisticRegression(random_state=21, fit_intercept=False, max_iter=1000)
cross_val_scores(logreg_baseline, X_train, y_train)

train - 0.62902   |   valid - 0.59259
train - 0.64633   |   valid - 0.62963
train - 0.63479   |   valid - 0.56296
train - 0.65622   |   valid - 0.61481
train - 0.63397   |   valid - 0.57778
train - 0.64056   |   valid - 0.59259
train - 0.64138   |   valid - 0.65926
train - 0.65952   |   valid - 0.56296
train - 0.64333   |   valid - 0.59701
train - 0.63674   |   valid - 0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943

CPU times: user 208 ms, sys: 7.15 ms, total: 215 ms
Wall time: 215 ms


(0.6016473189607519,
 0.029427270750289624,
 [0.5925925925925926,
  0.6296296296296297,
  0.562962962962963,
  0.6148148148148148,
  0.5777777777777777,
  0.5925925925925926,
  0.6592592592592592,
  0.562962962962963,
  0.5970149253731343,
  0.6268656716417911])

### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [6]:
penalties = ['none', 'l1', 'l2']

In [7]:
for pen in penalties:
    if pen == 'none':
        solver = 'lbfgs'
    else:
        solver = 'liblinear'
    logreg = LogisticRegression(
        penalty=pen, solver=solver, random_state=21,
        fit_intercept=False, max_iter=1000, multi_class='ovr'
    )
    print(f"Penalty = {pen}")
    cross_val_scores(logreg, X_train, y_train)

Penalty = none
train - 0.65293   |   valid - 0.62222
train - 0.65293   |   valid - 0.62963
train - 0.65128   |   valid - 0.57037
train - 0.66364   |   valid - 0.62963
train - 0.64468   |   valid - 0.59259
train - 0.64798   |   valid - 0.57778
train - 0.64963   |   valid - 0.66667
train - 0.67024   |   valid - 0.60741
train - 0.65486   |   valid - 0.60448
train - 0.64333   |   valid - 0.61940
Average accuracy on crossval is 0.61202
Std is 0.02670

Penalty = l1
train - 0.61830   |   valid - 0.54815
train - 0.62737   |   valid - 0.62222
train - 0.60511   |   valid - 0.54074
train - 0.63644   |   valid - 0.62222
train - 0.62407   |   valid - 0.55556
train - 0.62325   |   valid - 0.58519
train - 0.61253   |   valid - 0.63704
train - 0.64716   |   valid - 0.58519
train - 0.63015   |   valid - 0.59701
train - 0.61367   |   valid - 0.59701
Average accuracy on crossval is 0.58903
Std is 0.03129

Penalty = l2
train - 0.61006   |   valid - 0.56296
train - 0.61665   |   valid - 0.61481
train - 0.6

## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [8]:
%%time
svm_baseline = SVC(kernel='linear', probability=True, random_state=21)
cross_val_scores(svm_baseline, X_train, y_train)

train - 0.70486   |   valid - 0.65926
train - 0.69662   |   valid - 0.75556
train - 0.69415   |   valid - 0.62222
train - 0.70239   |   valid - 0.65185
train - 0.69085   |   valid - 0.65185
train - 0.68920   |   valid - 0.64444
train - 0.69250   |   valid - 0.72593
train - 0.70074   |   valid - 0.62222
train - 0.69605   |   valid - 0.61940
train - 0.71087   |   valid - 0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359

CPU times: user 1.75 s, sys: 13 ms, total: 1.76 s
Wall time: 1.76 s


(0.6587064676616916,
 0.043585708770590564,
 [0.6592592592592592,
  0.7555555555555555,
  0.6222222222222222,
  0.6518518518518519,
  0.6518518518518519,
  0.6444444444444445,
  0.725925925925926,
  0.6222222222222222,
  0.6194029850746269,
  0.6343283582089553])

### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [9]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]

In [10]:
for C in C_values:
    svm = SVC(kernel='linear', C=C, probability=True, random_state=21)
    print(f"C = {C}")
    cross_val_scores(svm, X_train, y_train)

C = 0.001
train - 0.23495   |   valid - 0.22963
train - 0.23495   |   valid - 0.22963
train - 0.23495   |   valid - 0.22963
train - 0.23495   |   valid - 0.22963
train - 0.23413   |   valid - 0.23704
train - 0.23413   |   valid - 0.23704
train - 0.23413   |   valid - 0.23704
train - 0.23413   |   valid - 0.23704
train - 0.23394   |   valid - 0.23881
train - 0.23394   |   valid - 0.23881
Average accuracy on crossval is 0.23443
Std is 0.00397

C = 0.01
train - 0.37923   |   valid - 0.40000
train - 0.37923   |   valid - 0.40000
train - 0.38417   |   valid - 0.35556
train - 0.35449   |   valid - 0.36296
train - 0.38252   |   valid - 0.37037
train - 0.38087   |   valid - 0.38519
train - 0.37923   |   valid - 0.40000
train - 0.38252   |   valid - 0.37037
train - 0.38468   |   valid - 0.35075
train - 0.38386   |   valid - 0.35821
Average accuracy on crossval is 0.37534
Std is 0.01848

C = 0.1
train - 0.58120   |   valid - 0.55556
train - 0.57543   |   valid - 0.56296
train - 0.57378   |   val

## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [11]:
%%time
tree_baseline = DecisionTreeClassifier(max_depth=10, random_state=21)
cross_val_scores(tree_baseline, X_train, y_train)

train - 0.81039   |   valid - 0.74074
train - 0.77741   |   valid - 0.74074
train - 0.83347   |   valid - 0.70370
train - 0.79720   |   valid - 0.76296
train - 0.82440   |   valid - 0.75556
train - 0.80379   |   valid - 0.68889
train - 0.80709   |   valid - 0.76296
train - 0.80132   |   valid - 0.65926
train - 0.80807   |   valid - 0.75373
train - 0.80478   |   valid - 0.68657
Average accuracy on crossval is 0.72551
Std is 0.03562

CPU times: user 36 ms, sys: 1.73 ms, total: 37.7 ms
Wall time: 37 ms


(0.7255113322277501,
 0.03562429627613885,
 [0.7407407407407407,
  0.7407407407407407,
  0.7037037037037037,
  0.762962962962963,
  0.7555555555555555,
  0.6888888888888889,
  0.762962962962963,
  0.6592592592592592,
  0.753731343283582,
  0.6865671641791045])

### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [12]:
depths = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]

In [13]:
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=21)
    print(f"max_depth = {depth}")
    cross_val_scores(tree, X_train, y_train)

max_depth = 2
train - 0.43034   |   valid - 0.47407
train - 0.43281   |   valid - 0.45185
train - 0.44023   |   valid - 0.38519
train - 0.43034   |   valid - 0.47407
train - 0.43776   |   valid - 0.40741
train - 0.43364   |   valid - 0.44444
train - 0.43364   |   valid - 0.44444
train - 0.43776   |   valid - 0.40741
train - 0.41763   |   valid - 0.38806
train - 0.43657   |   valid - 0.41791
Average accuracy on crossval is 0.42949
Std is 0.03116

max_depth = 4
train - 0.53998   |   valid - 0.48148
train - 0.52679   |   valid - 0.52593
train - 0.54493   |   valid - 0.47407
train - 0.54163   |   valid - 0.52593
train - 0.54493   |   valid - 0.57037
train - 0.52679   |   valid - 0.51111
train - 0.52844   |   valid - 0.50370
train - 0.53669   |   valid - 0.48889
train - 0.53624   |   valid - 0.54478
train - 0.54613   |   valid - 0.46269
Average accuracy on crossval is 0.50889
Std is 0.03190

max_depth = 6
train - 0.64880   |   valid - 0.57778
train - 0.62077   |   valid - 0.60000
train - 0.

In [14]:
print("Bonus: Trying min_samples_split = 5, min_samples_leaf = 3")
tree_tuned = DecisionTreeClassifier(
    max_depth=10, min_samples_split=5, min_samples_leaf=3, random_state=21
)
cross_val_scores(tree_tuned, X_train, y_train)

Bonus: Trying min_samples_split = 5, min_samples_leaf = 3
train - 0.78895   |   valid - 0.72593
train - 0.75021   |   valid - 0.71111
train - 0.80874   |   valid - 0.69630
train - 0.77824   |   valid - 0.72593
train - 0.80049   |   valid - 0.74074
train - 0.77906   |   valid - 0.68889
train - 0.78071   |   valid - 0.74074
train - 0.77576   |   valid - 0.63704
train - 0.78007   |   valid - 0.71642
train - 0.78171   |   valid - 0.67910
Average accuracy on crossval is 0.70622
Std is 0.03035



(0.7062189054726368,
 0.03034520007423509,
 [0.725925925925926,
  0.7111111111111111,
  0.6962962962962963,
  0.725925925925926,
  0.7407407407407407,
  0.6888888888888889,
  0.7407407407407407,
  0.6370370370370371,
  0.7164179104477612,
  0.6791044776119403])

## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [15]:
%%time
rf_baseline = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
cross_val_scores(rf_baseline, X_train, y_train)

train - 0.96455   |   valid - 0.88148
train - 0.96208   |   valid - 0.91852
train - 0.96785   |   valid - 0.86667
train - 0.96455   |   valid - 0.89630
train - 0.96538   |   valid - 0.91111
train - 0.96538   |   valid - 0.88148
train - 0.97115   |   valid - 0.91852
train - 0.96867   |   valid - 0.85185
train - 0.97364   |   valid - 0.88060
train - 0.97941   |   valid - 0.86567
Average accuracy on crossval is 0.88722
Std is 0.02204

CPU times: user 470 ms, sys: 9.56 ms, total: 480 ms
Wall time: 487 ms


(0.8872194582642343,
 0.022044865936245422,
 [0.8814814814814815,
  0.9185185185185185,
  0.8666666666666667,
  0.8962962962962963,
  0.9111111111111111,
  0.8814814814814815,
  0.9185185185185185,
  0.8518518518518519,
  0.8805970149253731,
  0.8656716417910447])

### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [16]:
for n_est in [20, 50, 100]:
    for depth in [10, 14, 20]:
        rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=21)
        print(f"n_estimators = {n_est}, max_depth = {depth}")
        cross_val_scores(rf, X_train, y_train)

n_estimators = 20, max_depth = 10
train - 0.82358   |   valid - 0.77037
train - 0.84996   |   valid - 0.82963
train - 0.86645   |   valid - 0.80741
train - 0.88788   |   valid - 0.80741
train - 0.86645   |   valid - 0.85926
train - 0.84666   |   valid - 0.74074
train - 0.85326   |   valid - 0.82222
train - 0.84996   |   valid - 0.75556
train - 0.88221   |   valid - 0.79104
train - 0.87068   |   valid - 0.76119
Average accuracy on crossval is 0.79448
Std is 0.03555

n_estimators = 20, max_depth = 14
train - 0.96373   |   valid - 0.87407
train - 0.93899   |   valid - 0.88148
train - 0.95713   |   valid - 0.85185
train - 0.95960   |   valid - 0.87407
train - 0.96043   |   valid - 0.91111
train - 0.96785   |   valid - 0.86667
train - 0.95548   |   valid - 0.88889
train - 0.95796   |   valid - 0.83704
train - 0.96293   |   valid - 0.89552
train - 0.95799   |   valid - 0.86567
Average accuracy on crossval is 0.87464
Std is 0.02023

n_estimators = 20, max_depth = 20
train - 0.99340   |   vali

In [17]:
print("Bonus: Trying min_samples_split = 5")
rf_tuned = RandomForestClassifier(
    n_estimators=50, max_depth=14, min_samples_split=5, random_state=21
)

Bonus: Trying min_samples_split = 5


In [18]:
cross_val_scores(rf_tuned, X_train, y_train)

train - 0.94394   |   valid - 0.86667
train - 0.94147   |   valid - 0.91111
train - 0.95383   |   valid - 0.84444
train - 0.95136   |   valid - 0.88148
train - 0.94971   |   valid - 0.91111
train - 0.94641   |   valid - 0.87407
train - 0.95054   |   valid - 0.90370
train - 0.94806   |   valid - 0.82222
train - 0.95964   |   valid - 0.87313
train - 0.94646   |   valid - 0.85075
Average accuracy on crossval is 0.87387
Std is 0.02800



(0.8738695411829738,
 0.02800414135261774,
 [0.8666666666666667,
  0.9111111111111111,
  0.8444444444444444,
  0.8814814814814815,
  0.9111111111111111,
  0.8740740740740741,
  0.9037037037037037,
  0.8222222222222222,
  0.8731343283582089,
  0.8507462686567164])

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [19]:
best_model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=21)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"\nFinal test accuracy: {test_acc:.5f}")


Final test accuracy: 0.93195


In [20]:
errors = (y_pred != y_test)

In [21]:
error_df = pd.DataFrame({'true': y_test, 'pred': y_pred, 'error': errors})

In [22]:
error_counts = error_df.groupby('true')['error'].sum()

In [23]:
total_counts = y_test.value_counts()

In [24]:
error_percent = (error_counts / total_counts * 100).fillna(0)
print("\nError percentage per weekday (0=Monday,...,6=Sunday):")
print(error_percent)


Error percentage per weekday (0=Monday,...,6=Sunday):
0    25.925926
1     9.090909
2     6.666667
3     2.500000
4    14.285714
5     5.555556
6     1.408451
dtype: float64


In [25]:
most_err_day = error_percent.idxmax()
print(f"Worst predicted weekday: {most_err_day} ({error_percent[most_err_day]:.2f}% errors)")

Worst predicted weekday: 0 (25.93% errors)


In [26]:
joblib.dump(best_model, '../data/best_model_regularization.joblib')
print("\nModel saved to '../data/best_model_regularization.joblib'")


Model saved to '../data/best_model_regularization.joblib'
